# 📏 Experiment: Impact of Different Length Distributions

This notebook implements the experiment to investigate the impact of mismatch between training and test set length distributions.

**Goal**: Train T5 models on 'Balanced' and 'Random' distributions and cross-evaluate them.
**Configuration**: Set `RUN_MODE` to choose which part of the experiment to run (useful for parallel Kaggle sessions).

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
RUN_MODE = "BALANCED"  # Options: "BALANCED", "RANDOM", "BOTH"
# ==========================================

import os
import shutil
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

CONFIG = {
    "model_name": "t5-small", # Change to t5-base for full experiment if resources allow
    "epochs": 50,
    "batch_size": 64,
    "accumulate_grad_batches": 4,
    "precision": "16-mixed",
    "check_val_every_n_epoch": 5,
    "train_samples": 100000,
    "val_samples": 10000,
    "test_samples": 10000,
    "max_digits": 30,
    "format": "10e-based",
    "seed": 42,
    "max_seq_length": 512,
    "output_dir": "outputs/length_experiment"
}

def run_command(command):
    print(f"⚡ Running: {command[:80]}...")
    exit_code = os.system(command)
    if exit_code != 0:
        print(f"❌ ERROR: Command failed with exit code {exit_code}")
        return False
    return True

In [ ]:
print("[1/5] 📁 Setup and Imports...")
# Copy files from /kaggle/input if needed (similar to original script)
if os.path.exists("/kaggle/input"):
    source_root = "/kaggle/input"
    dest_dir = "/kaggle/working"
    for root, dirs, files in os.walk(source_root):
        for file in files:
            if file.endswith(".py"):
                shutil.copy(os.path.join(root, file), os.path.join(dest_dir, file))

# Install dependencies
run_command("pip install -q pytorch-lightning transformers num2words protobuf==3.20.3 libraft-cu12>=25.6.0 rmm-cu12>=25.6.0")

In [ ]:
print("[2/5] 📊 Generating Data...")
run_command("mkdir -p data/length_exp outputs")

modes = ["balanced", "random"]
phases = [("train", CONFIG['train_samples']), ("val", CONFIG['val_samples']), ("test", CONFIG['test_samples'])]

for mode in modes:
    for phase, count in phases:
        output_file = f"data/length_exp/{phase}_{mode}.json"
        if os.path.exists(output_file):
            print(f"   Example {output_file} already exists, skipping.")
            continue
            
        print(f"   Generating {phase} set for {mode} ({count} samples)...")
        cmd = (
            f"python generate_data.py "
            f"--mode single "
            f"--count {count} "
            f"--digits {CONFIG['max_digits']} "
            f"--strategy {mode} "
            f"--format {CONFIG['format']} "
            f"--output {output_file} "
            f"--only_addition "
            f"--seed {CONFIG['seed']}"
        )
        if not run_command(cmd):
            raise RuntimeError(f"Failed to generate {output_file}")

In [ ]:
print(f"[3/5] 🏋️ Training (Mode: {RUN_MODE})...")

def train_model(mode_name):
    print(f"\n{'='*40}\nTRAINING: {mode_name.upper()}\n{'='*40}")
    
    train_file = f"data/length_exp/train_{mode_name}.json"
    val_file = f"data/length_exp/val_{mode_name}.json"
    test_file = f"data/length_exp/test_{mode_name}.json"
    output_path = f"{CONFIG['output_dir']}/{mode_name}"
    
    cmd = (
        f"python train.py "
        f"--train_file {train_file} "
        f"--val_file {val_file} "
        f"--test_file {test_file} "
        f"--output_dir {output_path} "
        f"--model_name_or_path {CONFIG['model_name']} "
        f"--format {CONFIG['format']} "
        f"--train_size_log {CONFIG['train_samples']} "
        f"--epochs {CONFIG['epochs']} "
        f"--train_batch_size {CONFIG['batch_size']} "
        f"--val_batch_size {CONFIG['batch_size']} "
        f"--lr 3e-4 "
        f"--seed {CONFIG['seed']} "
        f"--accelerator gpu "
        f"--devices 1 "
        f"--accumulate_grad_batches {CONFIG['accumulate_grad_batches']} "
        f"--precision {CONFIG['precision']} "
        f"--check_val_every_n_epoch {CONFIG['check_val_every_n_epoch']} "
        f"--max_seq_length {CONFIG['max_seq_length']}"
    )
    
    if not run_command(cmd):
        print(f"❌ Training for {mode_name} failed!")
    else:
        print(f"✅ Training for {mode_name} completed!")

if RUN_MODE in ["BALANCED", "BOTH"]:
    train_model("balanced")

if RUN_MODE in ["RANDOM", "BOTH"]:
    train_model("random")

In [ ]:
print("[4/5] 📝 Cross-Evaluation...")

def evaluate_model(train_mode, eval_mode):
    print(f"\n🔎 Evaluating Model trained on {train_mode.upper()} on {eval_mode.upper()} set...")
    
    # Locate best checkpoint
    ckpt_dir = f"{CONFIG['output_dir']}/{train_mode}"
    ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.ckpt')]
    if not ckpts:
        print(f"❌ No checkpoint found for {train_mode}")
        return
    
    # Assuming the first one is the best/last one saved
    ckpt_path = os.path.join(ckpt_dir, ckpts[0])
    print(f"   Using checkpoint: {ckpt_path}")
    
    test_file = f"data/length_exp/test_{eval_mode}.json"
    
    cmd = (
        f"python evaluate.py "
        f"--test_file {test_file} "
        f"--checkpoint_path \"{ckpt_path}\" "
        f"--output_dir {CONFIG['output_dir']}/{train_mode}_eval_on_{eval_mode} "
        f"--batch_size {CONFIG['batch_size']} "
        f"--seed {CONFIG['seed']}"
    )
    
    run_command(cmd)

# Evaluate metrics
if RUN_MODE == "BALANCED" or RUN_MODE == "BOTH":
    # Eval Balanced Model on Random Data (OOD)
    evaluate_model("balanced", "random")
    # Eval Balanced Model on Balanced Data (ID) - Optional since train does it, but good for check
    evaluate_model("balanced", "balanced")

if RUN_MODE == "RANDOM" or RUN_MODE == "BOTH":
    # Eval Random Model on Balanced Data (OOD)
    evaluate_model("random", "balanced")
    # Eval Random Model on Random Data (ID)
    evaluate_model("random", "random")

In [ ]:
print("[5/5] 📦 Zipping Results...")
run_command(f"zip -r results_length_exp_{RUN_MODE}.zip {CONFIG['output_dir']}")
print("✅ Done!")